# 09 — Cross-validation report assembly

This notebook runs the full `CrossValidationReport.write_all` on hand-constructed per-fold records and inspects the artefacts it produces: the aggregate markdown report, the per-split comparison reports, and the `cv_summary.json` payload. It confirms the report assembles the fold plan table, the training-across-folds table, and the per-split metric aggregates without touching `/ste/rnd`.

Modules exercised: `pipelines.cross_validation_pipeline.cv_report.CrossValidationReport`, `pipelines.cross_validation_pipeline.folds.FoldPlanner`, `pipelines.benchmark_pipeline.results.TrialRecord`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except Exception:
    torch = None

SEED = 0
RNG  = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.titlesize" : 11,
    "axes.labelsize" : 10,
    "image.cmap"     : "viridis",
})

print("Bootstrap complete. Repository root on sys.path:", Path("../..").resolve())


## Hand-construct base and per-split records

Base records carry checkpoint and training-result fields (used by the training-across-folds table); per-split records carry inference metrics.

In [ ]:
import json
import tempfile

from configuration.cross_validation_config import CrossValidationConfig
from pipelines.cross_validation_pipeline.folds import FoldPlanner
from pipelines.cross_validation_pipeline.cv_report import CrossValidationReport
from pipelines.benchmark_pipeline.results import TrialRecord
from tools.monitoring.logger import Logger

N_FOLDS = 4
config                     = CrossValidationConfig()
config.folds.n_folds       = N_FOLDS
config.folds.azimuth_start = 0
config.folds.azimuth_end   = 400
planner = FoldPlanner(config, range_start=0, range_end=100)

base_records = []
for fold_index in range(N_FOLDS):
    record                 = TrialRecord(name=f"fold_{fold_index}", run_dir=Path("/synthetic"))
    record.checkpoint      = {"best_val_loss": 0.20 + 0.01 * fold_index, "best_epoch": 30 + fold_index, "n_train_epochs": 80}
    record.training_result = {"duration_s": 1200 + 30 * fold_index}
    base_records.append(record)

def split_records(offset):
    records = []
    for fold_index in range(N_FOLDS):
        record = TrialRecord(name=f"fold_{fold_index}", run_dir=Path("/synthetic"))
        record.metrics = {
            "curve_rmse_gt"    : 0.50 + offset + 0.01 * fold_index,
            "overall_r2_gt"    : 0.90 - offset - 0.01 * fold_index,
            "pixel_r2_gt_mean" : 0.80 + 0.01 * fold_index,
        }
        records.append(record)
    return records

records_by_split = {"val": split_records(0.0), "test": split_records(0.02)}
print("records ready:", N_FOLDS, "folds, splits:", list(records_by_split))

## Run the real report assembly

In [ ]:
tmp_dir = Path(tempfile.mkdtemp())
logger  = Logger(log_dir=str(tmp_dir), name="cv_report_demo")
out_dir = tmp_dir / "reports"

report = CrossValidationReport(
    base_records     = base_records,
    records_by_split = records_by_split,
    planner          = planner,
    out_dir          = out_dir,
    model_name       = "resunet",
    embed_images     = False,
    logger           = logger,
)

written = report.write_all()
logger.close()

for path in written:
    print(path.relative_to(out_dir))

## Inspect the aggregate markdown report

We print the head of `cv_aggregate_report.md`, which should contain the fold plan table, the training-across-folds table, and the per-split metric aggregates.

In [ ]:
aggregate_md = (out_dir / "cv_aggregate_report.md").read_text(encoding="utf-8")
print("\n".join(aggregate_md.splitlines()[:48]))

## Inspect the structured summary payload

In [ ]:
summary = json.loads((out_dir / "cv_summary.json").read_text(encoding="utf-8"))
print("model   :", summary["model"])
print("n_folds :", summary["n_folds"])
print("folds   :", summary["folds"])
print("splits  :", list(summary["splits"]))
print("test curve_rmse_gt aggregate:", summary["splits"]["test"]["curve_rmse_gt"])

## Visualise per-split metric means from the summary payload

We read the mean and std straight back out of `cv_summary.json` and plot the val and test means side by side, confirming the payload carries the aggregated numbers.

In [ ]:
metrics = list(summary["splits"]["test"].keys())
x       = np.arange(len(metrics))
width   = 0.38

val_means  = [summary["splits"]["val"][m]["mean"]  for m in metrics]
val_stds   = [summary["splits"]["val"][m]["std"]   for m in metrics]
test_means = [summary["splits"]["test"][m]["mean"] for m in metrics]
test_stds  = [summary["splits"]["test"][m]["std"]  for m in metrics]

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(x - width / 2, val_means,  width, yerr=val_stds,  capsize=4, color="#f4c879", label="val")
ax.bar(x + width / 2, test_means, width, yerr=test_stds, capsize=4, color="#d96c6c", label="test")

ax.set_xticks(x)
ax.set_xticklabels(metrics, rotation=15, ha="right")
ax.set_ylabel("value")
ax.set_title("Per-split metric means from cv_summary.json")
ax.legend()
plt.tight_layout()
plt.show()

## Expected visual outcome

`write_all` returns a list of written paths including `cv_aggregate_report.md` and `cv_summary.json` plus per-split comparison files. The aggregate markdown head shows the fold plan and training tables, the summary payload reports the model, fold names, and per-split aggregates, and the grouped bar chart shows val and test means with std error bars for each metric.